# NegMerge Tutorial

## 1. Import Requirements

In [1]:
import torch
import json # JSON — a tool for reading and writing data in a structured format (like a neatly organised text file)
import argparse # Argument Parser — lets you pass settings to your program from the command line
import timm.data.transforms #TIMM is a popular library of pre-built AI vision models. This imports its image transformation/preprocessing tools.
import abc #Abstract Base Classes — a Python tool for creating "template" classes. (It's imported here but not visibly used yet.)
import sys #System tools — lets Python interact with the Python interpreter itself (e.g. modifying which folders Python looks in for code)
sys.path.insert(0, r'C:\Users\xl14n23\PycharmProjects\COMP6258-negmerge\CLIP_MU') # Python only looks for code in certain folders by default. This line says: "also look in this folder" — specifically a folder called CLIP_MU that contains custom code for this project.

import os # Operating System tools — lets Python talk to your computer's file system (finding files, reading folder paths, etc.)
os.environ["HF_CARS_ROOT"] = r"C:\Users\xl14n23\PycharmProjects\COMP6258-negmerge\datasets_local\stanford_cars_hf" #This sets an environment variable — essentially a labelled sticky note your program can read later. It's telling Python: "whenever I say HF_CARS_ROOT, I mean this folder". The dataset here is Stanford Cars, a popular image dataset of car photos used to train/test AI models.

# optional sanity check
import glob
print(glob.glob(os.path.join(os.environ["HF_CARS_ROOT"], "data", "train-*.parquet"))[:2])


C:\Users\xl14n23\PycharmProjects\COMP6258-negmerge\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['C:\\Users\\xl14n23\\PycharmProjects\\COMP6258-negmerge\\datasets_local\\stanford_cars_hf\\data\\train-00000-of-00001.parquet']


In [2]:
if 'ipykernel' in sys.modules:
    sys.argv = ['']

class MaybeToTensor:
    def __call__(self, x):
        return x
timm.data.transforms.MaybeToTensor = MaybeToTensor
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
import torch
print(torch.__version__)
print(torch.version.cuda)          # None if CPU-only build
print(torch.cuda.is_available())   # False

2.11.0+cu130
13.0
True


## 2. Define Configuration

In [4]:
# The whole function is just building that order form — listing every possible customisation the program accepts, and what the default should be if you don't specify it.
def parse_arguments():
    parser = argparse.ArgumentParser() # It is a container for argument specifications and has options that apply to the parser as whole
    # The ArgumentParser.add_argument() method attaches individual argument specifications to the parser.
    parser.add_argument("--data_location", type=str, default=os.path.expanduser("~/data"), help="The root directory for the datasets.")
    parser.add_argument("--eval-datasets", default=None, type=lambda x: x.split(","), help="Which datasets to use for evaluation. Split by comma, e.g. MNIST,EuroSAT.")
    parser.add_argument("--results_db", type=str, default=None, help="Where to store the results, else does not store")
    parser.add_argument("--model", type=str, default="ViT-B-32", help="The type of model (e.g. RN50, ViT-B-32).")
    parser.add_argument("--save", type=str, default=None, help="Optionally save a _classifier_, e.g. a zero shot classifier or probe.")
    parser.add_argument("--load", type=lambda x: x.split(","), default=None, help="Optionally load a _classifier_, e.g. a zero shot classifier or probe.")
    parser.add_argument("--seed", type=int, default=None, help="Random seed.")
    parser.add_argument("--finetuning_mode", choices=["standard", "linear", "none"], help="Whether to use linearized models or not.")
    parser.add_argument("--n-eval-points", type=int, default=21, help="Number of evaluation points used to find optimal coefficient in task arithmetic.")

    parsed_args = parser.parse_args() # reads the settings the user typed in the terminal and stores them all in parsed_args
    parsed_args.device = "cuda" if torch.cuda.is_available() else "cpu"

    if parsed_args.load is not None and len(parsed_args.load) == 1:
        parsed_args.load = parsed_args.load[0]
        
    return parsed_args


In [5]:
args = parse_arguments()

args.data_location = r"C:\Users\xl14n23\PycharmProjects\COMP6258-negmerge\datasets_local"
args.finetuning_mode = "standard"       # "linear" or "standard"
args.model = "ViT-B-32"                 # Backbone
args.results_db = "checkpoints"         # Names the folder where results will be saved.
args.save = os.path.join(args.results_db, args.finetuning_mode, args.model) # Builds a specific save path by joining three things together with `os.path.join`, So results go into: checkpoints/standard/ViT-B-32/
args.openclip_cachedir = os.path.expanduser("~/.cache/open_clip")

dataset = "Cars"                        # Forget set
control_dataset = "ImageNet"            # Retain set

FINETUNED_DIR = r"C:\Users\xl14n23\PycharmProjects\COMP6258-negmerge\Finetuned_them"

with open(os.path.join(FINETUNED_DIR, "zeroshot_accuracies.json")) as f: # zero-shot means the model was never specifically trained on that task, yet it can still attempt it.
    pretrained_accuracies = json.load(f) # Baseline: the original model's accuracy score.
negation_accuracies = {} # Creates an empty container (like a blank notebook page) that will be filled in later with the results of your forgetting experiment.

# Compatibility defaults expected by src/modeling.py and training/eval code
if not hasattr(args, "auto_aug"):
    args.auto_aug = None
if not hasattr(args, "train_dataset"):
    args.train_dataset = None
if not hasattr(args, "batch_size"):
    args.batch_size = 128 # Batch size of 128 based on the paper
if not hasattr(args, "num_workers"):
    args.num_workers = 8
if not hasattr(args, "cache_dir"):
    args.cache_dir = os.path.expanduser("~/.cache")


## 3. Dowload Pretrained and Fine-tuned Weights
- Download Link: https://drive.google.com/drive/u/1/folders/1m1iHi5KoTN1Fg5JqIZxtVP1ZTxgILZyi

In [6]:
pretrained_path = os.path.join(FINETUNED_DIR, 'zeroshot.pt')
finetuned_paths = [
    os.path.join(FINETUNED_DIR, f'clip-vit-b-32_cars_rand-m{m}-n{n}_finetuned.pt')
    for m in range(1, 11)
    for n in range(1, 4)
]


## 4. Define Task Vector Class

In [7]:
class _TaskVector(abc.ABC):
    def __init__(
        self, pretrained_checkpoint=None, finetuned_checkpoint=None, vector=None
    ):
        if vector is not None:
            self.vector = vector
        else:
            assert (
                pretrained_checkpoint is not None and finetuned_checkpoint is not None
            )
            with torch.no_grad():
                if isinstance(pretrained_checkpoint, dict):
                    pretrained_state_dict = pretrained_checkpoint
                else:
                    pretrained_state_dict = self._load_checkpoint(
                        pretrained_checkpoint
                    ).state_dict()

                if isinstance(finetuned_checkpoint, dict):
                    finetuned_state_dict = finetuned_checkpoint
                else:
                    finetuned_state_dict = self._load_checkpoint(
                        finetuned_checkpoint
                    ).state_dict()

                self.vector = {}
                for key in pretrained_state_dict:
                    if pretrained_state_dict[key].dtype == torch.int64:
                        continue
                    if pretrained_state_dict[key].dtype == torch.uint8:
                        continue
                    self.vector[key] = (
                        finetuned_state_dict[key] - pretrained_state_dict[key]
                    ) # Task vector

    @abc.abstractmethod
    def _load_checkpoint(self, checkpoint):
        """Load a checkpoint into a model."""
        raise NotImplementedError

    @abc.abstractmethod
    def _cast_to_same_type(self, other):
        raise NotImplementedError

    # Maths Operations
    def __add__(self, other):
        """Add two task vectors together."""
        other = self._cast_to_same_type(other)
        with torch.no_grad():
            new_vector = {}
            for key in self.vector:
                if key not in other.vector:
                    print(f"Warning, key {key} is not present in both task vectors.")
                    continue
                new_vector[key] = self.vector[key] + other.vector[key]
        return self.__class__(vector=new_vector)

    def __sub__(self, other):
        """Subtract two task vectors."""
        return self.__add__(-other)

    def __radd__(self, other):
        if other is None or isinstance(other, int):
            return self
        return self.__add__(other)

    def __neg__(self):
        """Negate a task vector."""
        with torch.no_grad():
            new_vector = {}
            for key in self.vector:
                new_vector[key] = -self.vector[key]
        return self.__class__(vector=new_vector)

    def __pow__(self, power):
        """Power of a task vector."""
        with torch.no_grad():
            new_vector = {}
            for key in self.vector:
                new_vector[key] = self.vector[key] ** power
        return self.__class__(vector=new_vector)

    def __mul__(self, other):
        """Multiply a task vector by a scalar."""
        with torch.no_grad():
            new_vector = {}
            for key in self.vector:
                new_vector[key] = other * self.vector[key]
        return self.__class__(vector=new_vector)

    def dot(self, other):
        """Dot product of two task vectors."""
        other = self._cast_to_same_type(other)
        with torch.no_grad():
            dot_product = 0.0
            for key in self.vector:
                if key not in other.vector:
                    print(f"Warning, key {key} is not present in both task vectors.")
                    continue
                dot_product += torch.sum(self.vector[key] * other.vector[key])
        return dot_product

    def norm(self):
        """Norm of a task vector."""
        return torch.sqrt(self.dot(self))

    # Applying the task vector
    def apply_to(self, pretrained_checkpoint, scaling_coef=1.0):
        """Apply a task vector to a pretrained model."""
        with torch.no_grad():
            pretrained_model = self._load_checkpoint(pretrained_checkpoint)
            new_state_dict = {}
            pretrained_state_dict = pretrained_model.state_dict()
            for key in pretrained_state_dict:
                if key not in self.vector:
                    print(
                        f"Warning: key {key} is present in the pretrained state dict but not in the task vector"  # noqa: E501
                    )
                    continue
                new_state_dict[key] = (
                    pretrained_state_dict[key] + scaling_coef * self.vector[key]
                )
        pretrained_model.load_state_dict(new_state_dict)
        return pretrained_model


class NonLinearTaskVector(_TaskVector):
    """A task vector for nonlinear models."""

    def _load_checkpoint(self, checkpoint):
        """Load a checkpoint into a model."""
        return torch.load(checkpoint, map_location="cuda" if torch.cuda.is_available() else "cpu", weights_only=False)

    def apply_to_nonlinear(self, pretrained_nonlinear_checkpoint, scaling_coef=1.0):
        """Apply a task vector to a nonlinear pretrained model."""
        return self.apply_to(pretrained_nonlinear_checkpoint, scaling_coef)
    
    def _cast_to_same_type(self, other):
        return linear_to_nonlinear(other, self.vector.keys())

def linear_to_nonlinear(linear_task_vector, param_names):
    """Convert a linear task vector to a nonlinear task vector."""
    if isinstance(linear_task_vector, NonLinearTaskVector):
        return linear_task_vector
    else:
        return NonLinearTaskVector(
            vector=linear_task_vector.get_named_parameters(param_names)
        )


## 5. Merge Task Vectors

In [8]:
for idx, finetuned_path in enumerate(finetuned_paths): # Look through all 30 fine-tuned model file paths one by one. idx: counter, finetuned_path: actual file path
    state_dict = torch.load(finetuned_path, map_location=device, weights_only=False) # Loads the current fine-tuned model file from disk into memory.
    state_dict = {k: v.to(device) for k, v in state_dict.items()}
        
    task_vector = (NonLinearTaskVector(pretrained_path, state_dict))

    if idx == 0: # Set up empty containers (for the first model only)
        merged_vector = {k: torch.zeros_like(v) for k, v in task_vector.vector.items()}
        mask = {k: torch.zeros_like(v) for k, v in task_vector.vector.items()}

    for key in task_vector.vector.keys():
        merged_vector[key] += task_vector.vector[key] # Sum
        mask[key] += torch.sign(task_vector.vector[key]) # Sign consensus, torch.sign: +1, -1, 0

for key in torch.load(finetuned_path, map_location=device, weights_only=False).keys():
    consistency_mask = torch.abs(mask[key]) == len(finetuned_paths)
    task_vector.vector[key] = torch.where(consistency_mask, merged_vector[key] / len(finetuned_paths), torch.zeros_like(merged_vector[key]))


## 6. Evaluate

### 6.1. Find Optimal Coefficient

In [9]:
import os
from datasets import load_dataset
from tqdm import tqdm
from huggingface_hub import get_token

OUT_ROOT = r"C:\Users\xl14n23\PycharmProjects\COMP6258-negmerge\datasets_local\imagenet"
os.makedirs(os.path.join(OUT_ROOT, "val"), exist_ok=True)

token = get_token()
if not token:
    raise RuntimeError(
        "No Hugging Face token found. Run `hf auth login` in your terminal first."
    )

# Validation only: streaming avoids downloading/validating train/test splits
try:
    val_ds = load_dataset(
        "ILSVRC/imagenet-1k",
        split="validation",
        token=token,
        streaming=True,
    )
except Exception as e:
    msg = str(e)
    if "403 Forbidden" in msg or "public gated repositories" in msg:
        raise RuntimeError(
            "HF token lacks gated-dataset permission. Enable 'Access to public gated repositories' "
            "for your token at https://huggingface.co/settings/tokens, then run `hf auth login` again."
        ) from e
    raise

def export_val(ds):
    split_dir = os.path.join  (OUT_ROOT, "val")
    os.makedirs(split_dir, exist_ok=True)

    for i, ex in enumerate(tqdm(ds, desc="Export val")):
        img = ex["image"]
        label = int(ex["label"])
        cls_dir = os.path.join(split_dir, f"{label:04d}")
        os.makedirs(cls_dir, exist_ok=True)
        img.save(os.path.join(cls_dir, f"{i:08d}.JPEG"), format="JPEG", quality=95)

export_val(val_ds)
print("Validation export done.")

Export val: 0it [00:00, ?it/s]'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/ILSVRC/imagenet-1k/resolve/49e2ee26f3810fb5a7536bbf732a7b07389a47b5/data/validation-00000-of-00014.parquet
Retrying in 1s [Retry 1/5].
Export val: 0it [00:15, ?it/s]


KeyboardInterrupt: 

In [10]:
from pathlib import Path
from datasets import load_dataset
import os, glob, sys

# Use consistent root-based path resolution (same logic as Cell 6)
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "CLIP_MU").exists() and (PROJECT_ROOT / "..").exists():
    for candidate in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
        if (candidate / "CLIP_MU").exists() and (candidate / "finetuned_them").exists():
            PROJECT_ROOT = candidate
            break

root = (PROJECT_ROOT / "datasets_local" / "stanford_cars_hf").resolve()
data_dir = root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

ds = load_dataset("tanganke/stanford_cars")

# Keep the splits your loader needs
ds["train"].to_parquet(str(data_dir / "train-00000-of-00001.parquet"))
ds["test"].to_parquet(str(data_dir / "test-00000-of-00001.parquet"))

os.environ["HF_CARS_ROOT"] = str(root)

print("HF_CARS_ROOT =", os.environ["HF_CARS_ROOT"])
print("train files:", glob.glob(str(data_dir / "train-*.parquet")))
print("test files:", glob.glob(str(data_dir / "test-*.parquet")))

Creating parquet from Arrow format: 100%|██████████| 10/10 [00:01<00:00,  5.77ba/s]

HF_CARS_ROOT = C:\Users\xl14n23\PycharmProjects\COMP6258-negmerge\datasets_local\stanford_cars_hf
train files: ['C:\\Users\\xl14n23\\PycharmProjects\\COMP6258-negmerge\\datasets_local\\stanford_cars_hf\\data\\train-00000-of-00001.parquet']
test files: ['C:\\Users\\xl14n23\\PycharmProjects\\COMP6258-negmerge\\datasets_local\\stanford_cars_hf\\data\\test-00000-of-00001.parquet']


In [11]:
#os.remove("checkpoints/standard/ViT-B-32/head_CarsVal.pt")
#print("Deleted cached head, it will rebuild on next evaluation.")

Deleted cached head, it will rebuild on next evaluation.


In [11]:
from PIL import Image
import glob

val_dir = "your_path"
corrupted = []

for fpath in glob.glob(val_dir + r"\**\*.JPEG", recursive=True):
    try:
        Image.open(fpath).verify()
    except Exception:
        corrupted.append(fpath)

print(len(corrupted))

0


In [13]:
from src.eval import evaluate_task_vector, evaluate_task_vector_at_coef
from src.utils import find_optimal_coef
print("loaded lib")
args.eval_datasets = [dataset + "Val"]
args.control_dataset = control_dataset + "Val"
val_metrics = evaluate_task_vector(
    -task_vector,
    pretrained_path,
    args,
)

optimal_coef = find_optimal_coef(
    val_metrics,
    metric=f"{dataset}Val:top1",
    minimize=True,
    control_metric=f"{control_dataset}Val:top1",
    control_metric_threshold=0.95 * pretrained_accuracies[control_dataset + "Val"],
)

# If no coefficient satisfies the control threshold, fall back to best forget metric.
if optimal_coef is None:
    optimal_coef = min(
        val_metrics.keys(),
        key=lambda c: val_metrics[c][f"{dataset}Val:top1"],
    )
    print(
        "No coefficient satisfied control threshold; "
        f"falling back to lowest {dataset}Val:top1 at coef={optimal_coef:.3f}"
    )

print(f"Selected optimal coefficient: {optimal_coef:.3f}")

loaded lib
Evaluating for scaling coefficient 0.00
Evaluating on CarsVal
Classification head for ViT-B-32 on CarsVal exists at checkpoints/standard/ViT-B-32/head_CarsVal.pt
Loading classification head from checkpoints/standard/ViT-B-32/head_CarsVal.pt


100%|██████████| 7/7 [01:39<00:00, 14.26s/it]


Done evaluating on CarsVal. Accuracy: 56.39%
CarsVal Top-1 accuracy: 0.5639
Evaluating on ImageNetVal
Classification head for ViT-B-32 on ImageNetVal exists at checkpoints/standard/ViT-B-32/head_ImageNetVal.pt
Loading classification head from checkpoints/standard/ViT-B-32/head_ImageNetVal.pt


100%|██████████| 391/391 [03:20<00:00,  1.95it/s]


Done evaluating on ImageNetVal. Accuracy: 61.56%
ImageNetVal Top-1 accuracy: 0.6156
Evaluating for scaling coefficient 0.05
Evaluating on CarsVal
Classification head for ViT-B-32 on CarsVal exists at checkpoints/standard/ViT-B-32/head_CarsVal.pt
Loading classification head from checkpoints/standard/ViT-B-32/head_CarsVal.pt


...

Done evaluating on ImageNetVal. Accuracy: 58.14%
ImageNetVal Top-1 accuracy: 0.5814
Evaluating for scaling coefficient 1.00
Evaluating on CarsVal
Classification head for ViT-B-32 on CarsVal exists at checkpoints/standard/ViT-B-32/head_CarsVal.pt
Loading classification head from checkpoints/standard/ViT-B-32/head_CarsVal.pt


100%|██████████| 7/7 [01:31<00:00, 13.02s/it]


Done evaluating on CarsVal. Accuracy: 20.88%
CarsVal Top-1 accuracy: 0.2088
Evaluating on ImageNetVal
Classification head for ViT-B-32 on ImageNetVal exists at checkpoints/standard/ViT-B-32/head_ImageNetVal.pt
Loading classification head from checkpoints/standard/ViT-B-32/head_ImageNetVal.pt


100%|██████████| 391/391 [04:29<00:00,  1.45it/s]

Done evaluating on ImageNetVal. Accuracy: 57.78%
ImageNetVal Top-1 accuracy: 0.5778
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.63326999

### 6.2. Evaluate on the test set with the optimal coefficient.

In [14]:
args.eval_datasets = [dataset]
args.control_dataset = control_dataset

control_eval_dataset = control_dataset
imagenet_train_dir = os.path.join(args.data_location, "imagenet", "train")
train_has_class_dirs = os.path.isdir(imagenet_train_dir) and any(
    os.path.isdir(os.path.join(imagenet_train_dir, d)) for d in os.listdir(imagenet_train_dir)
)
if control_dataset == "ImageNet" and not train_has_class_dirs:
    control_eval_dataset = "ImageNetVal"
    print(
        "ImageNet train split is missing/empty; using ImageNetVal as control dataset for test-time evaluation."
    )

args.control_dataset = control_eval_dataset

if optimal_coef is None:
    if "val_metrics" not in globals() or val_metrics is None:
        raise RuntimeError(
            "optimal_coef is None and val_metrics is unavailable. Re-run Cell 17 first."
        )
    optimal_coef = min(
        val_metrics.keys(),
        key=lambda c: val_metrics[c][f"{dataset}Val:top1"],
    )
    print(
        "No valid control-constrained coefficient in memory; "
        f"using fallback coef={optimal_coef:.3f} from validation metrics."
    )

test_metrics = evaluate_task_vector_at_coef(
    -task_vector,
    pretrained_path,
    args,
    optimal_coef,
)

print("=" * 100)
print(f"Test accuracy: {test_metrics[f'{dataset}:top1']}")

negation_accuracies[dataset] = {
    "test": test_metrics[f"{dataset}:top1"],
    "test_control": test_metrics[f"{control_eval_dataset}:top1"],
    "val": val_metrics,
}

print(negation_accuracies[dataset])

ImageNet train split is missing/empty; using ImageNetVal as control dataset for test-time evaluation.
Evaluating on Cars
Classification head for ViT-B-32 on CarsVal exists at checkpoints/standard/ViT-B-32/head_CarsVal.pt
Loading classification head from checkpoints/standard/ViT-B-32/head_CarsVal.pt


100%|██████████| 63/63 [01:37<00:00,  1.55s/it] 


Done evaluating on Cars. Accuracy: 21.23%
Cars Top-1 accuracy: 0.2123
Evaluating on ImageNetVal
Classification head for ViT-B-32 on ImageNetVal exists at checkpoints/standard/ViT-B-32/head_ImageNetVal.pt
Loading classification head from checkpoints/standard/ViT-B-32/head_ImageNetVal.pt


100%|██████████| 391/391 [04:19<00:00,  1.51it/s]

Done evaluating on ImageNetVal. Accuracy: 57.78%
ImageNetVal Top-1 accuracy: 0.5778
Test accuracy: 0.21228702897649546
{'test': 0.21228702897649546, 'test_control': 0.5777515550311006, 'val': {np.float64(0.0): {'CarsVal:top1': 0.5638820638820639, 'ImageNetVal:top1': 0.615592311846237}, np.float64(0.05): {'CarsVal:top1': 0.5405405405405406, 'ImageNetVal:top1': 0.6153123062461249}, np.float64(0.1): {'CarsVal:top1': 0.527027027027027, 'ImageNetVal:top1': 0.6146122922458449}, np.float64(0.15000000000000002): {'CarsVal:top1': 0.4975429975429975, 'ImageNetVal:top1': 0.6142122842456849}, np.float64(0.2): {'CarsVal:top1': 0.4705159705159705, 'ImageNetVal:top1': 0.6135522710454209}, np.float64(0.25): {'CarsVal:top1': 0.44717444717444715, 'ImageNetVal:top1': 0.6123122462449249}, np.float64(0.30000000000000004): {'CarsVal:top1': 0.42874692874692877, 'ImageNetVal:top1': 0.6113522270445408}, np.float64(0.35000000000000003): {'CarsVal:top1': 0.41277641277641275, 'ImageNetVal:top1': 0.610072201444028